# Treinamento do Bitcoin

Este notebook executa o mesmo treinamento do script: sete fechamentos para prever o próximo dia. É a alternativa permitida pelo enunciado quando o Docker não está disponível. Desenvolvido com apoio do Codex. Não é recomendação de investimento.

In [1]:
from pathlib import Path
import os
import sys
import json
import pandas as pd
import joblib

root = Path.cwd()
if not (root / 'data/btc_daily.csv').exists():
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
data = pd.read_csv('data/btc_daily.csv')
print('Dias:', len(data))
print('Período:', data['date'].iloc[0], 'até', data['date'].iloc[-1])
data.tail(7)

Dias: 1096
Período: 2023-01-01 até 2025-12-31


,date,close
1089,2025-12-25,87172.0
1090,2025-12-26,87296.0
1091,2025-12-27,87810.0
1092,2025-12-28,87870.0
1093,2025-12-29,87119.0
1094,2025-12-30,88390.0
1095,2025-12-31,87496.0


## Treinar e exportar

O script monta as janelas em ordem cronológica, separa 80% para treino e 20% para teste, ajusta a regressão e exporta o modelo. A referência é repetir o último fechamento. A implementação fica num único lugar para o notebook e o Docker usarem o mesmo código.

In [2]:
from training.train import main
main()

CSV: 1096 dias; exemplos completos: 1089
{
  "linear_regression": {
    "mae_usd": 1518.282399469047,
    "rmse_usd": 2013.7607131244888
  },
  "last_close_baseline": {
    "mae_usd": 1480.2293577981652,
    "rmse_usd": 1968.3752159651085
  },
  "train_examples": 871,
  "test_examples": 218
}
Alvos de treino: 2023-01-08 a 2025-05-27
Alvos de teste: 2025-05-28 a 2025-12-31
Modelo exportado e recarregado com as mesmas previsões.


## Carregar o arquivo em seguida

Aqui o modelo é aberto pelo arquivo exportado. A entrada termina em 31/12/2025; o próximo dia é 01/01/2026. Essa previsão não demonstra execução do backend em container.

In [3]:
from features import make_input
model = joblib.load('artifacts/model.joblib')
metadata = json.loads(Path('artifacts/metadata.json').read_text(encoding='utf-8'))
prices = data['close'].tail(7).tolist()
predicted = round(float(model.predict(make_input(prices))[0]), 2)
print('Modelo salvo:', metadata['model'])
print('Fechamentos:', prices)
print('Previsão para 01/01/2026 em USD:', predicted)
assert metadata['train_target_end'] < metadata['test_target_start']
assert predicted > 0
print('Exportação e recarga verificadas.')

Modelo salvo: LinearRegression
Fechamentos: [87172.0, 87296.0, 87810.0, 87870.0, 87119.0, 88390.0, 87496.0]
Previsão para 01/01/2026 em USD: 87703.54
Exportação e recarga verificadas.
